# TV-Denoising Network: Hybrid Deep-Learning & Convex Optimization

A PyTorch implementation of **Differentiable Total Variation Denoising** using `cvxpylayers`. 

`cvxpylayers` is a package developed by Agrawal et al. and introduced in ["Differentiable Convex Optimization Layers"](https://arxiv.org/abs/1910.12449). It allows convex problems that are **DPP-compliant**, meaning they are constructed following the **Disciplined Parametrized Programming** grammar, to be integrated into a Neural Network as a native layer. This enables end-to-end backpropagation through the optimization solver using implicit differentiation.

In this notebook, we load pre-trained deep-learning models containing optimization layers to study different approaches to Total Variation (TV) Denoising on the DIV2K dataset. The dataset is converted to grayscale, and we manually add Additive White Gaussian Noise using the `src.data.dataset.GaussianDenoisingDataset` utility.

### Model Architecture
The model is a hybrid system composed of:
* **CNN Backbone (Weight Predictor):** Analyzes local image features and noise levels to predict a spatially-adaptive $\Lambda$ (lambda) map.
* **CVXPY Optimization Layer:** Solves the TV proximal operator using the predicted $\Lambda$ map as the regularization weight for each pixel.

The backbone outputs a regularization map corresponding to either **"isotropic"** or **"anisotropic"** Total Variation penalties.

### Training Objectives
Models are trained using a **dual-loss** function: 
$$\mathcal{L} = \alpha \cdot \text{MSE} + (1 - \alpha) \cdot (1 - \text{SSIM})$$

This ensures the model does not merely learn to match the ground truth pixel-by-pixel (MSE) but also maintains structural integrity and perceptual quality by mimicking the human visual system (HVS).

---

### Experimental Configurations
Training was conducted across several configurations to evaluate solver efficiency and denoising quality:

1.  **Anisotropic Regularization** with the **SCS** (First-order Splitting Conic) solver.
2.  **Anisotropic Regularization** with the **CLARABEL** (Interior Point Method) solver.
3.  **Isotropic Regularization** with the **SCS** solver.
4.  **Isotropic Regularization** with the **CLARABEL** solver.

#### Numerical Observations
Due to the mathematical coupling of horizontal and vertical gradients in the isotropic formulation (solved as a Second-Order Cone Program), **isotropic training is effectively 5 to 20 times slower** than anisotropic training (solved as a Quadratic Program). Consequently, isotropic models were trained for fewer iterations. 

The slowdown observed during isotropic training is due to the L2 regularization introduced that complexify the KKT system resolution during backward passes, this is shown at the end of the notebook.

Furthermore, during isotropic training, we observed that the model struggled to converge initially. This necessitated specific architectural adjustments, including weight initialization of the CNN backbone and architectural changes to handle patches' boundaries, to facilitate stable convergence. However, althought results are slightly better, the loss seemed to stagnate after 5 epochs for isotropic regularization based models.

Finally, no matter the model be always observe oscillating loss during training despite changing a lot the learning rate (for gradient vanish problems) but in vain. 

#### Hardware used for training

The whole training was performed using an Apple Sillicon M4 Pro chip. Pytorch does have a backend for M chips' `gpu` called `mps`, however the `cvxpylayer` layer performs certain matrix factorizations that are not handled by that backend. Therefore, the forward pass in the Optimization layer **only** was done on the `cpu`.
This does not come with high performance problem from repeatedly switching from `gpu` memory to `cpu` memory since for Apple Sillicon chips, they share the same RAM, however while the memory is physically shared, the overhead of synchronization and the dispatch latency between the GPU (CNN backbone) and CPU (Solver) is significant in PyTorch, (cf. benchmarks below).

#### Training not part of notebook

Due to the long time needed for training (1 model could take up to dozens of hours), I have decided not to include training part in the notebook. If the reader wishes to train the model or tryout other configurations, please refer to: (https://github.com/Kh-T5/Diff-TV-Net) & the corresponding README.

## Imports

In [ ]:
import os
import sys
notebook_dir = os.path.abspath('')
project_root = os.path.dirname(notebook_dir)

if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
### project related imports
from src.models.nn_hybrid import TVDenoisingNet
from src.data.dataset import GaussianDenoisingDataset
from src.utils.evaluation import evaluate, save_debug_plot
from src.config import (
    gaussian_std,
    BENCHMARK_PATCH_SIZE,
    scs_info,
    CLARABEL_info
)

### Visualization imports 
import matplotlib.pyplot as plt 
import pandas as pd
import numpy as np
import time
import seaborn as sns

### Pytorch imports
import torch
from torch.utils.data import DataLoader
import torch.nn as nn

### Convex Optimization imports
import cvxpy as cp

### Metrics
import torch.nn.functional as F
from skimage.metrics import structural_similarity as ssim
from piq import ssim as piq_ssim


## Model loading

In [ ]:
### Samples
samples_dir = r'../data/sample/benchmarking_samples/'
### Models
clarabel_iso = r"../results/models/isotropic/model_clarabel_solver_isotropic.pth"
clarabel_ani = r"../results/models/anisotropic/model_clarabel_solver_anisotropic.pth"
scs_iso = r"../results/models/isotropic/model_scs_solver_low_init_isotropic.pth"
scs_ani = r"../results/models/anisotropic/model_scs_solver_anisotropic.pth"

In [ ]:
def load_benchmarking_model(reg_type, solver_name, img_size=(BENCHMARK_PATCH_SIZE, BENCHMARK_PATCH_SIZE), device="cpu"):
    """
    Loads one of the 4 models given the regularization type and the solver name.
    """
    configs = {
        "anisotropic_scs": {"reg": "anisotropic", "solver": "SCS", "file": scs_ani},
        "anisotropic_clarabel": {"reg": "anisotropic", "solver": "CLARABEL", "file": clarabel_ani},
        "isotropic_scs": {"reg": "isotropic", "solver": "SCS", "file": scs_iso},
        "isotropic_clarabel": {"reg": "isotropic", "solver": "CLARABEL", "file": clarabel_iso},
    }
    
    solver_params = {
        "SCS": scs_info,
        "CLARABEL": CLARABEL_info
    }

    model = TVDenoisingNet(
        reg=reg_type, 
        solver=solver_name, 
        solver_info=solver_params[solver_name],
        img_size=img_size
    )
    path = configs[f"{reg_type.lower()}_{solver_name.lower()}"]["file"]
    state_dict = torch.load(path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval() 
        

        
    return model

## Training evolution

In [ ]:
def plot_training_history(histories, plots_dir, reg_type, model_name, val_interval=10):
    epochs = len(histories["loss"])
    train_epochs = np.arange(1, epochs + 1)
    val_epochs = np.arange(val_interval, epochs + 1, val_interval)

    fig, axs = plt.subplots(3, 1, figsize=(10, 12), sharex=True)
    fig.suptitle(f"Training Progress: {model_name} ({reg_type})", fontsize=16, fontweight='bold')
    axs[0].plot(train_epochs, histories["loss"], label=r'Training Loss ($\mathcal{L}$)', color='royalblue', lw=2)
    axs[0].set_ylabel(r'Loss')
    axs[0].set_title(r'Evolution of Training Loss')
    axs[0].grid(True, alpha=0.3)
    axs[0].legend()

    axs[1].plot(val_epochs, histories["mse"], label=r'Val $MSE$', color='darkorange', marker='o', ls='--')
    axs[1].set_ylabel(r'$MSE$')
    axs[1].set_title(r'Validation Mean Squared Error')
    axs[1].grid(True, alpha=0.3)
    axs[1].legend()

    axs[2].plot(val_epochs, histories["ssim"], label=r'Val $SSIM$', color='forestgreen', marker='s', ls='--')
    axs[2].set_ylabel(r'$SSIM$')
    axs[2].set_xlabel(r'Epoch')
    axs[2].set_title(r'Validation Structural Similarity Index')
    axs[2].set_ylim(0, 1.05)
    axs[2].grid(True, alpha=0.3)
    axs[2].legend()

    plt.tight_layout()
    path = os.path.join(plots_dir, f"{reg_type}/{model_name}.png")
    plt.savefig(path)
    plt.show()



In [ ]:
plots_dir = r"../results/plots/"
histories_scs_anisotropic = np.load(r"../results/models/anisotropic/result_scs_solver_anisotropic.npz")
histories_clarabel_anisotropic = np.load(r"../results/models/anisotropic/result_clarabel_solver_isotropic.npz")
plot_training_history(histories_scs_anisotropic, plots_dir, "anisotropic", "SCS")
plot_training_history(histories_clarabel_anisotropic, plots_dir, "anisotropic", "CLARABEL", val_interval=5)

### Comments

These figures illustrate the comparative training dynamics between the **SCS** and **CLARABEL** solvers integrated into the **Diff-TV-Net** architecture. Both models were trained using an anisotropic Total Variation penalty.

### Training Loss Evolution ($\mathcal{L}$)
* **SCS (Non-Initialized Weights):** The loss exhibits a sharp, "hard" decrease within the first 5 epochs, dropping from $\sim 0.065$ to $\sim 0.044$. This represents the CNN backbone rapidly shifting from its random initial state to a more structured feature extraction regime. After epoch 10, the loss plateaus around $0.041$, characterized by oscillations.
* **CLARABEL (Initialized Weights):** Because the CNN weights were pre-initialized, the model begins at a lower loss baseline ($\sim 0.044$). The descent is does not have the same "hard" descent pattern. We still observe the oscillations. 



### Interpretations

When looking at the denoised output, below in the notebook or in the README file, we see the staircase effect emerge in the image. 

* This is known to be a direct cause of the **anisotorpic** regluratization independancy between the gradient components. This could be the cause of our stagnation observations during training for both solvers. 

* The **isotropic** approach may solve this issue, however due to hardware limitations, running training on 50 epochs was not reasonable and during the **5/10 epochs** they were trained on, both solvers seemed to stagnate aswell..


## Benchmark utils

### Benchmark functions for baseline solvers and our trained models

Since `cvxpylayers` performs canonicalization once during initialization, the forward pass is theoretically much faster than standard CVXPY for repeated solves.

The utils functions/classes below serve different purposes:
- Evaluate solve time between base and cnn backbone models.
- Analysis of the CNN learnt lambda map.
- Visualization of outputs and lambda maps of Baseline VS CNN-backbone equivalent
- Evaluating metrics on sample unseen data: Baseline VS CNN-backbone equivalent
- Evaluating correlation between grad and lambda maps for CNN-backbone models.
- Evaluating forward and backward time spent by the CNN-backbone models

In [ ]:
class TVBenchmarker:
    """
    Benchmarking class, used to compare baseline solvers from cvxpy 
    with the saved trained models using cvxpylayers and pytorch
    """
    def __init__(self, reg_type: str, solver_name: str, device_name="cpu"):
        self.device = device_name if torch.backends.mps.is_available() else "cpu"
        self.model = load_benchmarking_model(reg_type, solver_name)
        self.model.eval()
        self.model.to(self.device)
        self.reg = reg_type
        self.solver_name = solver_name

        self.solver_map = {"SCS": cp.SCS, "CLARABEL": cp.CLARABEL}
        self.solver_params = {
            "SCS": scs_info,
            "CLARABEL": CLARABEL_info
        }
        solver_info = self.solver_params[solver_name.upper()]
        self.filtered_info = {k: v for k, v in solver_info.items() if k != 'solve_method'}
        
    def run_pure_cvxpy(self, noisy, lam_scalar):
        """Standard CVXPY: modeling + canonicalization + solve."""
        if torch.is_tensor(noisy):
            noisy_np = noisy.squeeze(0).squeeze(0).cpu().numpy()
        else:
            noisy_np = noisy
        
        h, w = noisy_np.shape
        U = cp.Variable((h, w))
        
        if self.reg == "anisotropic":
            ux = U[1:, :] - U[:-1, :]
            uy = U[:, 1:] - U[:, :-1]
            reg_term = cp.sum(cp.abs(ux)) + cp.sum(cp.abs(uy))
        elif self.reg == "isotropic":
            reg_term = cp.tv(U)
        else:
            raise KeyError(f"Reg not recognized : {self.reg}")
            
        prob = cp.Problem(cp.Minimize(0.5 * cp.sum_squares(U - noisy_np) + lam_scalar * reg_term))
        
        start = time.perf_counter()
        prob.solve(solver=self.solver_map[self.solver_name], **self.filtered_info)
        elapsed = time.perf_counter() - start
        
        return U.value, elapsed

    def run_hybrid(self, noisy_tensor):
        """Hybrid: CNN inference + pre-compiled cvxpylayer."""
        start = time.perf_counter()
        noisy_tensor = noisy_tensor.to(self.device)
        with torch.no_grad():
            denoised, lam_map = self.model(noisy_tensor)
        elapsed = time.perf_counter() - start
        
        return denoised.squeeze(0).cpu().numpy(), lam_map.squeeze(0).cpu().numpy(), elapsed

In [ ]:
def compute_metrics(output, target):
    """
    Computes mse, psnr and ssim for a given clean, denoised couple of numpy arrays
    """
    if torch.is_tensor(output): 
        output = output.detach().cpu().numpy()
    if torch.is_tensor(target): 
        target = target.detach().cpu().numpy()

    output = np.squeeze(output)
    target = np.squeeze(target)

    if output.shape != target.shape:
        h, w = output.shape
        target = target[:h, :w]

    mse = np.mean((target - output) ** 2)
    psnr = 10 * np.log10(1.0 / mse) if mse > 0 else 100
    
    ssim_val = ssim(target, output, data_range=1.0)
    
    return mse, psnr, ssim_val

In [ ]:
def evaluate_model(reg: str, solver: str, lam: float, dataloader: DataLoader, device="cpu", n_iter=20):
    """
    Compares a baseline model with its CNN-backbone equivalent.

    Iterates over the sample dataset (5 samples from DIV2K test dataset) 
    and returns average metrics values.
    Inputs:
        - reg: str, regularization
        - solver: str, solver
        - lam: float, regularization factor for baseline solver.
        - dataloader: pytorch.data.utils.Dataloader, sample data loader 
        - device: str, device to use in pytorch
        - n_iter: int, number of epochs on the 5-images sample dataset
    """
    mses, psnrs, ssims = [], [], []
    cnn_mses, cnn_psnrs, cnn_ssims = [], [], []

    benchmarker = TVBenchmarker(reg, solver)
    with torch.no_grad():
        for _ in range(n_iter):
            for noisy, clean in dataloader:
                noisy, clean = noisy.to(device), clean.to(device)
                clean_np = clean.squeeze(0).squeeze(0).cpu().numpy()
                noisy_np = noisy.squeeze(0).squeeze(0).cpu().numpy()

                output_cnn, lam_map, elapsed_cnn = benchmarker.run_hybrid(noisy)
                output_base, elapsed_cnn = benchmarker.run_pure_cvxpy(noisy_np, lam)

                
                
                mse, psnr, ssim_val = compute_metrics(output_cnn, clean_np)
                cnn_mses.append(mse)
                cnn_psnrs.append(psnr)
                cnn_ssims.append(ssim_val)

                mse, psnr, ssim_val = compute_metrics(output_base, clean_np)
                mses.append(mse)
                psnrs.append(psnr)
                ssims.append(ssim_val)




    return {
        "MSE base": np.mean(mses),
        "PSNR base": np.mean(psnrs),
        "SSIM base": np.mean(ssims),

        "MSE cnn": np.mean(cnn_mses),
        "PSNR cnn": np.mean(cnn_psnrs),
        "SSIM cnn": np.mean(cnn_ssims)
    }

In [ ]:
def benchmark_lambda_map(reg, solver, dataloader, num_iter = 20, device="cpu"):
    """ 
    Computes metrics on the lambda maps learnt by the model given regularization and solver to use.
    Computes correlations between lambda_maps output and image gradients.
    Inputs:
        - reg: str, regularization
        - solver: str, solver
        - dataloader: pytorch.data.utils.Dataloader, sample data loader 
        - device: str, device to use in pytorch
        - num_iter: int, number of epochs on the 5-images sample dataset

    """
    def gradient_magnitude(img):
        dx = img[:, :, :, 1:] - img[:, :, :, :-1]
        dy = img[:, :, 1:, :] - img[:, :, :-1, :]
        
        dx = F.pad(dx, (0, 1, 0, 0)) 
        dy = F.pad(dy, (0, 0, 0, 1))
        
        return torch.sqrt(dx**2 + dy**2)
    
    lambda_maps = []
    corrs = []

    model = load_benchmarking_model(reg, solver, device=device)
    for _ in range(num_iter):
        with torch.no_grad():
            for noisy, _ in dataloader:
                denoised, lam_map = model(noisy.to(device))
                grad_mag = gradient_magnitude(noisy)
                lambda_maps.append(lam_map.cpu())
                corrs.append(torch.corrcoef(
            torch.stack([lam_map.flatten(), grad_mag.flatten()])
        )[0,1].item())
    
    lam_all = torch.cat(lambda_maps)

    print("Mean λ:", lam_all.mean().item())
    print("Std λ:", lam_all.std().item())
    print("Mean λ-gradient correlation:", np.mean(corrs))

In [ ]:
def benchmark_forward_backward(reg, solver, dataloader, alpha, n_iter=5):
    """
    Evaluates time taken for forward and backward pass for the specified regularization and solver.
    Inputs:
        - reg: str, regularization
        - solver: str, solver
        - dataloader: pytorch.data.utils.Dataloader, sample data loader 
        - alpha: float, dual-loss factor
        - n_iter: int, number of epochs on the 5-images sample dataset
    """
    fwd_times = []
    bwd_times = []
    device = torch.device("mps" if torch.mps.is_available() else "cpu")

    model = load_benchmarking_model(reg, solver)
    model = model.to(device)
    model.train()
    def sync():
        if device.type == "mps":
            torch.mps.synchronize()

    for _ in range(n_iter):
        for noisy, clean in dataloader:
            noisy = noisy.to(device)
            clean = clean.to(device)
            noisy.requires_grad_(True)

            sync()
            start = time.perf_counter()

            denoised, _ = model(noisy)
            sync()
            mid = time.perf_counter()

            denoised_crop = denoised[..., 2:-2, 2:-2]
            clean_crop = clean[..., 2:-2, 2:-2]

            mse_loss = torch.nn.functional.mse_loss(denoised, clean)
            ssim_val = piq_ssim(denoised_crop, clean_crop, data_range=1.0)
            loss = alpha * mse_loss + (1 - alpha) * (1 - ssim_val)

            loss.backward()
            sync()
            end = time.perf_counter()

            fwd_times.append(mid - start)
            bwd_times.append(end - mid)

            model.zero_grad()
            noisy.grad = None
    
    mean_fwd_time, mean_bwd_time = np.mean(fwd_times), np.mean(bwd_times)
    print(f"Mean forward pass time for {reg} regularization and solver {solver}: {float(mean_fwd_time):.3f} s")
    print(f"Mean backward pass time for {reg} regularization and solver {solver}: {float(mean_bwd_time):.3f} s")
    return mean_fwd_time, mean_bwd_time

### plot utils

In [ ]:
def run_global_inference_benchmark(dataloader, lam_scalar, device="cpu", n_iter=3):
    """
    Compares forward pass time for all configurations between CNN-backbone model VS. baseline counterpart.
    """
    configs = [
        ("anisotropic", "SCS"),
        ("anisotropic", "CLARABEL"),
        ("isotropic", "SCS"),
        ("isotropic", "CLARABEL")
    ]
    
    all_results = []

    for reg, solver in configs:
        print(f"--- Benchmarking {reg.upper()} with {solver} ---")
        
        benchmarker = TVBenchmarker(reg, solver, device)
        
        h_times = []
        b_times = []

        for _ in range(n_iter):
            for noisy, _ in dataloader:
                _, _, t_h = benchmarker.run_hybrid(noisy)
                h_times.append(t_h)

                _, t_b = benchmarker.run_pure_cvxpy(noisy, lam_scalar)
                b_times.append(t_b)

        res = {
            "Regularization": reg.capitalize(),
            "Solver": solver,
            "Hybrid Mean (s)": np.mean(h_times),
            "Baseline Mean (s)": np.mean(b_times),
            "Speedup (x)": np.mean(b_times) / np.mean(h_times)
        }
        all_results.append(res)
        print(f"Speedup: {res['Speedup (x)']:.2f}x\n")

In [ ]:
def visualize_benchmark_results(
        dataloader: DataLoader, 
        reg_type: str, 
        solver_name: str, 
        global_lam: float, 
        num_samples: int=5
        ):
    """
    Visualizes the comparison between Hybrid and Baseline.
    
    Inputs:
        model: The TVDenoisingNet instance.
        dataloader: DataLoader returning (noisy, clean) batches.
        reg_type: "anisotropic" or "isotropic".
        solver_name: "SCS" or "CLARABEL".
        global_lam: Scalar lambda for the standard CVXPY baseline.
    """
    device = "cpu"
    model = load_benchmarking_model(reg_type=reg_type, solver_name=solver_name)
    model.eval()

    
    solver_info = scs_info if solver_name.lower() == "scs" else CLARABEL_info
    cp_solver= cp.SCS if solver_name.upper() == "SCS" else cp.CLARABEL
    filtered_info = {k: v for k, v in solver_info.items() if k != 'solve_method'}
    print(solver_info, solver_name)

    fig, axes = plt.subplots(num_samples, 5, figsize=(20, 4 * num_samples))
    plt.subplots_adjust(wspace=0.15, hspace=0.3)
    
    cols = ["Ground Truth", "Noisy Input", f"Hybrid ({reg_type})", r"Learned $\Lambda$ Map", "Baseline"]

    with torch.no_grad():
        for i, (noisy, clean) in enumerate(dataloader):
            if i >= num_samples:
                break
            
            noisy, clean = noisy.to(device), clean.to(device)
            
            denoised_h, lam_map = model(noisy)
            
            noisy_np = noisy[0, 0].cpu().numpy()
            U = cp.Variable(noisy_np.shape)
            
            if reg_type == "anisotropic":
                ux = U[1:, :] - U[:-1, :]
                uy = U[:, 1:] - U[:, :-1]
                reg_term = cp.sum(cp.abs(ux)) + cp.sum(cp.abs(uy))
            elif reg_type == "isotropic":
                reg_term = cp.tv(U)
            else:
                raise KeyError(f"Regularization {reg_type} not recognized.")
            
            prob = cp.Problem(cp.Minimize(0.5 * cp.sum_squares(U - noisy_np) + global_lam * reg_term))
            prob.solve(solver=cp_solver, **filtered_info)
            denoised_b = U.value

            display_list = [
                clean[0, 0].cpu().numpy(),
                noisy[0, 0].cpu().numpy(),
                denoised_h[0, 0].cpu().numpy(),
                lam_map[0].cpu().numpy(),
                denoised_b
            ]

            for j, img in enumerate(display_list):
                ax = axes[i, j] if num_samples > 1 else axes[j]
                
                cmap = 'magma' if j == 3 else 'gray'
                im = ax.imshow(img, cmap=cmap)
                
                if j == 3:
                    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
                
                if i == 0:
                    ax.set_title(cols[j], fontweight='bold', fontsize=12)
                ax.axis('off')

    plt.show()

In [ ]:
def plot_full_benchmarks(results_list):
    df = pd.DataFrame(results_list)
    df['Config'] = df['Reg'] + "\n(" + df['Solver'] + ")"

    fig, axs = plt.subplots(1, 3, figsize=(20, 6))
    sns.set_style("whitegrid")

    metrics = [
        ("MSE", r"Mean Squared Error ($MSE$)", 'rocket_r'),
        ("PSNR", r"Peak Signal-to-Noise Ratio ($PSNR$)", 'mako'),
        ("SSIM", r"Structural Similarity ($SSIM$)", 'viridis')
    ]

    for i, (col, label, palette) in enumerate(metrics):
        ax = axs[i]
        sns.barplot(data=df, x='Config', y=col, hue='Method', ax=ax, palette=palette)
        
        ax.set_title(label, fontweight='bold', fontsize=14)
        ax.set_xlabel("")
        ax.set_ylabel("")
        
        fmt = '.4f' if col == 'MSE' else '.2f'
        for p in ax.patches:
            if p.get_height() > 0:
                ax.annotate(format(p.get_height(), fmt), 
                            (p.get_x() + p.get_width() / 2., p.get_height()), 
                            ha='center', va='center', xytext=(0, 9), 
                            textcoords='offset points', fontsize=9, fontweight='bold')

    plt.suptitle("Comparative Performance: Hybrid Diff-TV-Net vs. Global Baseline", 
                 fontsize=16, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()

## Benchmark

**Sample dataset for benchmark:** 5 high resolution images from the test DIV2K dataset. All models are limited to 64x64 patches of those images for inference due to hardware limitations for higher resolution patches.

- GLOBAL_LAMBDA for baseline solver was selected with cross-validation on part of the training dataset.
- ALPHA corresponds to the dual-loss factor between mse and ssim. I used the same alpha throughout training.
- N_ITER is the number of epochs used in statistical estimations.
- N_ITER_FWD_BWD is similar to N_ITER for computation savings

In [ ]:
dataset = GaussianDenoisingDataset(samples_dir, patch_size=BENCHMARK_PATCH_SIZE, sigma=gaussian_std)
loader = DataLoader(dataset, batch_size=1, shuffle=False) 
GLOBAL_LAMBDA = 0.0505
ALPHA = 0.8
N_ITER = 20
N_ITER_FWD_BWD = 5

### SCS and CLARABEL SOLVER Visual Comparison: CNN backbone trained model VS BASELINE


The following grid provides a qualitative evaluation of our **Hybrid Diff-TV-Net** against a standard **CVXPY Baseline**. Each row represents a sample from the DIV2K validation set, processed with the following five-column structure:

1.  **Ground Truth**: The original, noise-free grayscale image.
2.  **Noisy Input**: The image after applying Additive White Gaussian Noise ($\sigma=0.1$).
3.  **Hybrid Model**: The denoised result from our network, which uses a CNN-predicted $\Lambda$ map to solve the Total Variation proximal operator.
4.  **Learned $\Lambda$ Map**: A heatmap representing the spatially-adaptive regularization weights predicted by the CNN. Darker regions indicate lower $\lambda$ values, showing where the network "relaxes" smoothing to preserve high-contrast edges and textures.
5.  **Baseline**: The denoised result using a standard CVXPY solve with a fixed, global regularization parameter ($\lambda=0.1$).


**Note to reader**: Images' patches are drawn randomly by the dataloader, if a given "ground truth" patch does not look like much during visualization, try running the cell again. 

#### **Anisotropic regularization**

In [ ]:
visualize_benchmark_results(loader, "anisotropic", "SCS", GLOBAL_LAMBDA, num_samples=3)
visualize_benchmark_results(loader, "anisotropic", "CLARABEL", GLOBAL_LAMBDA, num_samples=3)

#### **Isotropic regularization**

In [ ]:
visualize_benchmark_results(loader, "isotropic", "SCS", GLOBAL_LAMBDA, num_samples=3)
visualize_benchmark_results(loader, "isotropic", "CLARABEL", GLOBAL_LAMBDA, num_samples=3)

### Metrics benchmarks

In here, we compare metrics on the output provided by the CNN-backbone models VS their baseline solvers

Learned vs. Global Prior: The Hybrid model consistently outperforms the baseline in SSIM, even if the MSE remains comparable. This is expected in bi-level optimization: the MSE (L2 loss) is often easy for a global solver to minimize, but the structural integrity (SSIM) requires the spatially-adaptive Λ map to avoid the "staircasing" effect in smooth gradients.

**Isotropic Gain**: The gap between Hybrid and Baseline is larger in the Isotropic case. Because Isotropic TV is more computationally complex (SOCP), the CNN's ability to "relax" the cone constraint at edges is more impactful here than in the simpler Anisotropic (QP) case.

**Anisotropic outperforming Isotropic**: This is probably due to the difference in training. Anisotorpic training as mentioned above was complete and robust whereas for the isotropic case, only a handful of epochs were trained upon.

In [ ]:
configs = [
    ("anisotropic", "SCS"),
    ("anisotropic", "CLARABEL"),
    ("isotropic", "SCS"),
    ("isotropic", "CLARABEL")
]

results_list = []

for reg, solver in configs:
    print(f"Evaluating {reg} with {solver}...")
    res = evaluate_model(reg, solver, GLOBAL_LAMBDA, loader, n_iter=N_ITER)
    
    results_list.append({
        "Method": "Hybrid",
        "Reg": reg.capitalize(),
        "Solver": solver,
        "MSE": res["MSE cnn"],
        "PSNR": res["PSNR cnn"],
        "SSIM": res["SSIM cnn"]
    })
    
    results_list.append({
        "Method": "Baseline",
        "Reg": reg.capitalize(),
        "Solver": solver,
        "MSE": res["MSE base"],
        "PSNR": res["PSNR base"],
        "SSIM": res["SSIM base"]
    })

plot_full_benchmarks(results_list)

**Conclusion:** The superiority of the Hybrid approach is most significant in the **Isotropic** case. Because Isotropic TV is a Second-Order Cone Program (SOCP), the sensitivity to the regularization parameter is higher; here, the learned $\Lambda$ map provides pixel-wise relaxation of the TV penalty to maintain structural fidelity.

SSIM is expected to be higher for CNN-backbone models since they were trained on dual-loss including ssim, but the other results show that it does not mean our models do not outperform base ones.

### Learnt lambda map & correlation with gradients.


* **Negative Correlation :** The CNN correctly predicts **low ** at edges (high gradients) to preserve detail and **high ** in flat areas to remove noise.

* **Anisotropic vs. Isotropic:**
    * **Anisotropic :** Stronger correlation. The network must modulate  tightly to prevent "staircasing" artifacts.
    * **Isotropic :** Higher Mean. This penalty is naturally smoother, so the model compensates with higher overall regularization strength.


* **Solver Impact:** **CLARABEL** yields the most stable results (lowest Std ). Its high-precision interior-point method allows the network to learn a more refined, less noisy regularization map than the first-order **SCS** solver.

**Conclusion:** The non-zero standard deviation and consistent negative correlation confirm the model is not just acting as a global filter, but is actively using the CNN to "guide" the solver.

In [ ]:
print("\n------------------- Anisotropic SCS -----------------------\n")
benchmark_lambda_map("anisotropic", "SCS", loader, num_iter=N_ITER)

print("\n------------------- Anisotropic CLARABEL -----------------------\n")
benchmark_lambda_map("anisotropic", "CLARABEL", loader, num_iter=N_ITER)

print("\n------------------- Isotropic SCS -----------------------\n")
benchmark_lambda_map("isotropic", "SCS", loader, num_iter=N_ITER)

print("\n------------------- Isotropic CLARABEL -----------------------\n")
benchmark_lambda_map("isotropic", "CLARABEL", loader, num_iter=N_ITER)

### Forward pass time comparison: CNN-model VS baseline counterpart


The hybrid models should outperform their counterpart because `cvxpylayers` optimization layer only canonicalizes the problem at the initialization whereas the counterparts performa canonicalization every new patch they receive. 

However, a **Speedup < 1.0** indicates that the computational overhead of the CNN backbone and the PyTorch-to-CVXPY interface currently outweighs the benefits of pre-canonicalization...

In [ ]:
run_global_inference_benchmark(loader, GLOBAL_LAMBDA, device="cpu", n_iter=N_ITER_FWD_BWD)


In [ ]:
run_global_inference_benchmark(loader, GLOBAL_LAMBDA, device="mps", n_iter=N_ITER_FWD_BWD)


### Impact of the different solvers / regularization during training

The backward pass is consistently orders of magnitude slower than the forward pass. This is because backpropagating through a TV-solver requires solving a large, sparse KKT linear system derived from the Implicit Function Theorem

---
#### The CLARABEL Advantage
* **SCS (First order method):** While slightly faster in the forward pass, it suffers in the backward pass. Its lower precision can lead to ill-conditioned KKT matrices, making the implicit gradient solve computationally expensive (especially in the isotropic case at 17.2s).

* **CLARABEL (interior method):**  Although the forward pass is ~30% slower, the backward pass for Anisotropic TV is 3.3x faster than SCS. Interior point methods factorize the KKT matrix as part of the solve; This probably helps `cvxpylayers` that can reuse these factorizations, making the backward pass more efficient.

#### Regularization Complexity: L1 vs L2 norms

The transition from Anisotropic to Isotropic regularization in the backward pass is represented by much slower backward pass :

* **Anisotropic (Quadratic Program)**: The KKT system is highly sparse, keeping the backward pass time around 3 s.

* **Isotropic (Second-Order Cone Program):** The L2 constraints introduced in this regularization seem to over-complexify the resolution to the KKT system. This results could explain the slower training regime highlighted at the top of the notebook

##### Anisotropic regularization

In [ ]:
mean_fwd, mean_bwd = benchmark_forward_backward("anisotropic", "SCS", loader, alpha=ALPHA, n_iter=N_ITER_FWD_BWD)

In [ ]:
mean_fwd, mean_bwd = benchmark_forward_backward("anisotropic", "CLARABEL", loader, alpha=ALPHA, n_iter=N_ITER_FWD_BWD)

##### Isotorpic regularization

In [ ]:
mean_fwd, mean_bwd = benchmark_forward_backward("isotropic", "SCS", loader, alpha=ALPHA, n_iter=N_ITER_FWD_BWD)

In [ ]:
mean_fwd, mean_bwd = benchmark_forward_backward("isotropic", "CLARABEL", loader, alpha=ALPHA, n_iter=N_ITER_FWD_BWD)